# Entrenar reconocimiento de autopartes en Google Colab

Notebook autocontenido: incluye los módulos auditados del proyecto. No necesita subir el repositorio ni ejecutar NestJS/FastAPI. Descarga el dataset original con KaggleHub, entrena **YOLO11n-cls** y exporta los pesos y métricas reales.

1. Abre este archivo en [Google Colab](https://colab.research.google.com/).
2. Selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** (T4 es suficiente como punto inicial).
3. Ejecuta las celdas en orden. La GPU gratuita depende de la disponibilidad de tu cuenta.
4. Descarga el ZIP final y compártelo para integrarlo en el servicio local.

No se incluyen resultados previos. Los valores que aparezcan se obtendrán de tu ejecución. El test se evalúa **después** de seleccionar pesos mediante validación; no ajustes parámetros mirando test. La confianza de una predicción no garantiza una identificación correcta.

Referencias: [dataset Kaggle](https://www.kaggle.com/datasets/gpiosenka/car-parts-40-classes), [clasificación Ultralytics](https://docs.ultralytics.com/tasks/classify/), [YOLO11](https://docs.ultralytics.com/models/yolo11/).

## 1. Comprobar la GPU existente

Esta celda utiliza PyTorch preinstalado por Colab; no lo reinstala. Si no hay GPU disponible, cambia el entorno antes de continuar.

In [ ]:
import subprocess
import torch

print("PyTorch:", torch.__version__)
print("CUDA PyTorch:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU CUDA. En Colab selecciona Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU y vuelve a ejecutar.")
print("GPU:", torch.cuda.get_device_name(0))
print("Memoria GPU (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.run(["nvidia-smi"], check=True)


## 2. Instalar únicamente dependencias de entrenamiento

El resolvedor conserva torch/torchvision existentes si son compatibles. Si informa una incompatibilidad, reinicia con un entorno Colab actualizado; no continúes con un runtime parcialmente instalado.

In [ ]:
import sys
from pathlib import Path
from importlib.metadata import version
constraints = Path("/content/colab-existing-torch.txt")
constraints.write_text("torch==" + version("torch") + "\ntorchvision==" + version("torchvision") + "\n", encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-c", str(constraints), "ultralytics>=8.3,<9", "kagglehub>=0.3,<1", "Pillow>=11,<13", "scikit-learn>=1.5,<2"], check=True)
import torchvision
print("torchvision:", torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA dejó de estar disponible: reinicia un runtime GPU y revisa las dependencias.")


## 3. Crear los módulos del proyecto y revisar parámetros

Se escriben los módulos embebidos en `/content/autoparts-ai`. Los parámetros iniciales son **15 épocas, 224×224, patience 4 y batch automático GPU**. Puedes cambiar `epochs`, `patience`, `imgsz` o `batch` antes de entrenar. Si falta memoria, usa un batch entero menor, por ejemplo 16, 8 o 4. `workers=2` limita procesos de carga.

Las traducciones al español son un mapa manual revisable; no son descripciones originales del dataset.

In [ ]:
import json
from pathlib import Path

WORK = Path("/content/autoparts-ai")
EMBEDDED = {'autoparts_ai/dataset.py': '"""Inspect the original Kaggle files without inventing class information."""\nimport csv\nimport hashlib\nimport json\nimport os\nimport shutil\nfrom pathlib import Path\nfrom PIL import Image\n\nROOT = Path(__file__).resolve().parents[1]\nSLUG = "gpiosenka/car-parts-40-classes"\nEXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}\nSPLITS = {"train": ("train", "training"), "val": ("valid", "validation", "val"), "test": ("test", "testing")}\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")\n\n\ndef images(directory):\n    return sorted(p for p in Path(directory).rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONS)\n\n\ndef inspect_dataset(path, translations=None):\n    path = Path(path).resolve()\n    download_root = path\n    translations = translations or {}\n    candidates = []\n    for train_dir in [p for p in [path, *path.rglob("*")] if p.is_dir() and p.name.lower() in SPLITS["train"]]:\n        parent = train_dir.parent\n        class_names = sorted(p.name for p in train_dir.iterdir() if p.is_dir() and images(p))\n        partitions = {}\n        coherent = bool(class_names)\n        for split, aliases in SPLITS.items():\n            dirs = [p for p in parent.iterdir() if p.is_dir() and p.name.lower() in aliases]\n            if len(dirs) > 1:\n                coherent = False\n            if dirs:\n                partitions[split] = dirs[0]\n                if sorted(p.name for p in dirs[0].iterdir() if p.is_dir()) != class_names:\n                    coherent = False\n        coherent = coherent and "train" in partitions and "val" in partitions\n        candidates.append({"root": parent, "class_count": len(class_names), "coherent": coherent, "splits": list(partitions)})\n    valid = [c for c in candidates if c["coherent"]]\n    if valid:\n        largest = max(c["class_count"] for c in valid)\n        selected = [c for c in valid if c["class_count"] == largest]\n        if len(selected) > 1:\n            raise ValueError("Variantes equivalentes ambiguas. Use --dataset-root con una carpeta concreta.")\n        path = selected[0]["root"]\n    split_dirs = {}\n    for split, aliases in SPLITS.items():\n        matches = [p for p in [path, *path.rglob("*")] if p.is_dir() and p.name.lower() in aliases and any(c.is_dir() for c in p.iterdir())]\n        if len(matches) > 1:\n            raise ValueError(f"Partición {split} ambigua: {matches}")\n        if matches:\n            split_dirs[split] = matches[0]\n    if "train" not in split_dirs or "val" not in split_dirs:\n        raise ValueError("Se requieren train y valid/val separados; nunca se utiliza test como validación.")\n    names = sorted(p.name for p in split_dirs["train"].iterdir() if p.is_dir())\n    if not names:\n        raise ValueError("No se encontraron clases en train")\n    splits = {}\n    seen = {}\n    duplicate_content = []\n    for split, directory in split_dirs.items():\n        actual = sorted(p.name for p in directory.iterdir() if p.is_dir())\n        if actual != names:\n            raise ValueError(f"Clases diferentes en {split}")\n        counts = {}\n        for name in names:\n            files = images(directory / name)\n            if not files:\n                raise ValueError(f"Clase vacía: {split}/{name}")\n            for file in files:\n                with Image.open(file) as image:\n                    image.verify()\n                digest = hashlib.sha256(file.read_bytes()).hexdigest()\n                if digest in seen and seen[digest][0] != split:\n                    duplicate_content.append({"first": seen[digest], "second": [split, str(file.relative_to(path))]})\n                seen[digest] = (split, str(file.relative_to(path)))\n            counts[name] = len(files)\n        splits[split] = {"path": str(directory), "image_count": sum(counts.values()), "counts": counts}\n    files = [p for p in path.rglob("*") if p.is_file() and p.suffix.lower() not in EXTENSIONS]\n    documents, csv_files, licenses = [], [], []\n    for file in files:\n        relative = str(file.relative_to(path))\n        if file.suffix.lower() == ".csv":\n            with file.open(encoding="utf-8-sig", newline="") as stream:\n                reader = csv.DictReader(stream)\n                rows = list(reader)\n            csv_files.append({"file": relative, "columns": reader.fieldnames, "row_count": len(rows), "rows": rows})\n        elif file.suffix.lower() in {".txt", ".md", ".json", ".yaml", ".yml"}:\n            documents.append({"file": relative, "text": file.read_text(encoding="utf-8", errors="replace")[:50000]})\n        if "license" in file.name.lower() or "licence" in file.name.lower():\n            licenses.append(relative)\n    class_metadata = {}\n    for name in names:\n        associated = []\n        for table in csv_files:\n            columns = table["columns"] or []\n            label_columns = [column for column in columns if column.strip().lower() in {"label", "labels", "class", "class name", "class_name", "class names"}]\n            rows = [row for row in table["rows"] if any(row.get(column) == name for column in label_columns)]\n            if rows:\n                # Preserve original columns and values; never treat CSV image paths as descriptions.\n                fields = {}\n                for column in columns:\n                    if "path" in column.lower() or "file" in column.lower():\n                        continue\n                    values = sorted({row[column] for row in rows if row.get(column)})\n                    if len(values) <= 20:\n                        fields[column] = values\n                associated.append({"source_file": table["file"], "matching_row_count": len(rows), "fields": fields})\n        class_metadata[name] = {"csv": associated} if associated else {}\n    version = next((ancestor.name for ancestor in (download_root, *download_root.parents) if ancestor.parent.name == "versions"), None)\n    return {\n        "schema_version": 1,\n        "source": {"slug": SLUG, "url": f"https://www.kaggle.com/datasets/{SLUG}", "download_path": str(download_root), "selected_path": str(path), "selection_rule": "largest_coherent_classification_variant", "variants": [{"relative_root": str(c["root"].relative_to(download_root)), "class_count": c["class_count"], "coherent": c["coherent"], "splits": c["splits"]} for c in candidates], "version": version, "license": None, "license_files": licenses, "documents": documents},\n        "splits": splits, "class_count": len(names), "total_images": sum(s["image_count"] for s in splits.values()),\n        "classes": [{"id": i, "original_name": name, "training_label": name, "spanish_name": translations.get(name), "translation_source": "manual_reviewable_mapping" if translations.get(name) else None, "image_counts": {s: v["counts"][name] for s, v in splits.items()}, "metadata": class_metadata[name]} for i, name in enumerate(names)],\n        "original_files": [str(p.relative_to(path)) for p in files], "csv_metadata": csv_files,\n        "cross_split_duplicate_content": duplicate_content,\n    }\n\n\ndef download_dataset(output=ROOT / "artifacts/dataset.json", dataset_root=None):\n    output = Path(output)\n    if dataset_root is not None:\n        path = Path(dataset_root)\n    elif output.exists():\n        previous = json.loads(output.read_text(encoding="utf-8"))\n        candidate = Path(previous["source"]["download_path"])\n        if candidate.exists():\n            path = candidate\n        else:\n            import kagglehub\n            path = Path(kagglehub.dataset_download(SLUG))\n    else:\n        import kagglehub\n        path = Path(kagglehub.dataset_download(SLUG))\n    mapping = json.loads((ROOT / "config/translations.json").read_text(encoding="utf-8"))\n    manifest = inspect_dataset(path, mapping)\n    write_json(output, manifest)\n    print(f"Dataset: {path}; clases: {manifest[\'class_count\']}; imágenes: {manifest[\'total_images\']}")\n    return manifest\n\n\ndef prepare_dataset(manifest, destination=ROOT / "artifacts/prepared", splits=None):\n    destination = Path(destination).resolve()\n    # Fail closed: a previously prepared dataset is reused only with identical manifest.\n    marker = destination / "manifest.json"\n    chosen = {s: d for s, d in manifest["splits"].items() if splits is None or s in splits}\n    digest = hashlib.sha256(json.dumps({"manifest": manifest, "splits": sorted(chosen)}, sort_keys=True).encode()).hexdigest()\n    if destination.exists() and any(destination.iterdir()):\n        if marker.exists() and json.loads(marker.read_text())["source_sha256"] == digest:\n            return destination\n        raise ValueError("Destino preparado contiene otros datos. Use un destino nuevo; no se sobrescribe.")\n    for split, details in chosen.items():\n        for name in details["counts"]:\n            source = Path(details["path"]) / name\n            target = destination / split / name\n            target.mkdir(parents=True, exist_ok=True)\n            for index, file in enumerate(images(source)):\n                output = target / f"{index:07d}{file.suffix.lower()}"\n                try:\n                    os.link(file, output)\n                except OSError:\n                    shutil.copy2(file, output)\n    write_json(marker, {"source_sha256": digest, "dataset": manifest})\n    return destination\n', 'autoparts_ai/training.py': 'import csv\nimport hashlib\nimport json\nimport shutil\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom .dataset import ROOT, download_dataset, prepare_dataset, write_json\n\n\ndef validate_config(config):\n    permitted = {"model", "epochs", "patience", "imgsz", "batch", "device", "workers", "seed", "deterministic", "cache", "lr0", "optimizer", "dropout", "fraction"}\n    unknown = set(config) - permitted\n    if unknown:\n        raise ValueError(f"Parámetros no permitidos: {sorted(unknown)}")\n    for key in ("epochs", "imgsz"):\n        if isinstance(config.get(key), bool) or not isinstance(config.get(key), int) or config[key] <= 0:\n            raise ValueError(f"{key} debe ser entero positivo")\n    if not isinstance(config.get("patience"), int) or config["patience"] < 0:\n        raise ValueError("patience debe ser entero no negativo")\n    batch = config.get("batch", 16)\n    if isinstance(batch, bool) or not isinstance(batch, (int, float)) or not (batch == -1 or 0 < batch < 1 or isinstance(batch, int) and batch > 0):\n        raise ValueError("batch debe ser entero positivo, -1 o fracción de memoria (0,1)")\n    if config.get("model", "yolo11n-cls.pt") != "yolo11n-cls.pt":\n        raise ValueError("Este módulo entrena YOLO11n-cls con pesos preentrenados yolo11n-cls.pt")\n    return config\n\n\ndef train(config_path=ROOT / "config/train.json", manifest_path=ROOT / "artifacts/dataset.json"):\n    import torch\n    from ultralytics import YOLO\n    manifest_path = Path(manifest_path)\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else download_dataset(manifest_path)\n    if "train" not in manifest["splits"] or "val" not in manifest["splits"]:\n        raise ValueError("Se requieren train y val independientes; test nunca sustituye val")\n    # Ultralytics only sees train/val: test is physically absent from this view.\n    prepared = prepare_dataset(manifest, ROOT / "artifacts/training-data", splits={"train", "val"})\n    config = validate_config(json.loads(Path(config_path).read_text(encoding="utf-8")))\n    pretrained = config.pop("model", "yolo11n-cls.pt")\n    if config.get("device", "auto") == "auto":\n        config["device"] = 0 if torch.cuda.is_available() else "cpu"\n    if config["device"] == "cpu" and (not isinstance(config.get("batch", 16), int) or config.get("batch", 16) < 1):\n        config["batch"] = 16\n    model = YOLO(pretrained, task="classify")\n    best_snapshot = {}\n\n    def remember_saved_best(trainer):\n        # Ultralytics saves best.pt when current fitness equals best_fitness,\n        # including ties. Recording that event survives checkpoint stripping.\n        if trainer.fitness == trainer.best_fitness:\n            best_snapshot.update(epoch=int(trainer.epoch) + 1, fitness=float(trainer.fitness), validation_metrics={key: float(value) for key, value in trainer.metrics.items()})\n\n    model.add_callback("on_model_save", remember_saved_best)\n    started = time.monotonic()\n    model.train(data=str(prepared), project=str(ROOT / "artifacts/runs"), name="autoparts", **config)\n    elapsed = time.monotonic() - started\n    run = Path(model.trainer.save_dir)\n    best = run / "weights/best.pt"\n    if not best.is_file():\n        raise RuntimeError("Entrenamiento terminó sin best.pt")\n    best_model = YOLO(str(best), task="classify")\n    names = [c["original_name"] for c in manifest["classes"]]\n    if [best_model.names[i] for i in range(len(best_model.names))] != names:\n        raise RuntimeError("Orden de clases del modelo no coincide con dataset")\n    destination = ROOT / "artifacts/models"\n    destination.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(best, destination / "best.pt")\n    rows = []\n    if (run / "results.csv").exists():\n        with (run / "results.csv").open(newline="") as stream:\n            rows = [{k.strip(): v.strip() for k, v in row.items()} for row in csv.DictReader(stream)]\n    info = {"model": "YOLO11n-cls", "task": "classification", "created_at": datetime.now(timezone.utc).isoformat(), "class_count": len(names), "classes": manifest["classes"], "dataset": manifest["source"], "validation_metrics": best_snapshot.get("validation_metrics"), "training": {"parameters": config, "pretrained_weights": pretrained, "duration_seconds": elapsed, "run_directory": str(run), "best_epoch": best_snapshot.get("epoch"), "best_fitness": best_snapshot.get("fitness"), "epoch_metrics": rows, "image_counts": {s: d["image_count"] for s, d in manifest["splits"].items()}, "cross_split_duplicate_count": len(manifest.get("cross_split_duplicate_content", []))}, "metrics": None, "weights_sha256": hashlib.sha256(best.read_bytes()).hexdigest()}\n    write_json(destination / "model-info.json", info)\n    print(f"Pesos reales guardados: {destination / \'best.pt\'} ({elapsed:.1f}s). Ejecutar evaluate.py para métricas test.")\n    return info\n', 'autoparts_ai/evaluation.py': '"""Evaluate only after training, using the preserved holdout test set."""\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom .dataset import ROOT, images, write_json\n\n\ndef evaluate(weights=ROOT / "artifacts/models/best.pt", manifest_path=ROOT / "artifacts/dataset.json", device="cpu"):\n    from ultralytics import YOLO\n    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report\n    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))\n    if "test" not in manifest["splits"]:\n        raise ValueError("Dataset sin test separado: no se inventan métricas de prueba.")\n    model = YOLO(str(weights), task="classify")\n    info_path = Path(weights).parent / "model-info.json"\n    info = json.loads(info_path.read_text(encoding="utf-8")) if info_path.exists() else {"classes": manifest["classes"], "dataset": manifest["source"], "model": "YOLO11n-cls"}\n    image_size = info.get("training", {}).get("parameters", {}).get("imgsz", 224)\n    names = [c["original_name"] for c in manifest["classes"]]\n    if [model.names[i] for i in range(len(model.names))] != names:\n        raise ValueError("IDs del modelo no corresponden a las clases del manifiesto")\n    truth, predictions, top5 = [], [], []\n    test = Path(manifest["splits"]["test"]["path"])\n    for class_id, name in enumerate(names):\n        files = images(test / name)\n        for start in range(0, len(files), 16):\n            results = model.predict(source=[str(p) for p in files[start:start + 16]], imgsz=image_size, device=device, verbose=False)\n            for result in results:\n                truth.append(class_id)\n                predictions.append(result.probs.top1)\n                top5.append(class_id in result.probs.top5)\n    precision, recall, f1, _ = precision_recall_fscore_support(truth, predictions, average="macro", zero_division=0)\n    duplicates = manifest.get("cross_split_duplicate_content", [])\n    metrics = {"split": "test", "sample_count": len(truth), "top1_accuracy": float(accuracy_score(truth, predictions)), "top5_accuracy": sum(top5) / len(top5), "macro_precision": float(precision), "macro_recall": float(recall), "macro_f1": float(f1), "confusion_matrix": confusion_matrix(truth, predictions, labels=list(range(len(names)))).tolist(), "class_names": names, "per_class": classification_report(truth, predictions, labels=list(range(len(names))), target_names=names, output_dict=True, zero_division=0), "data_quality": {"cross_split_duplicate_count": len(duplicates), "holdout_independence_verified": not duplicates, "warning": "Existen imágenes idénticas entre particiones: estas métricas pueden sobreestimar generalización." if duplicates else None}}\n    info.update(metrics=metrics, weights_sha256=hashlib.sha256(Path(weights).read_bytes()).hexdigest())\n    write_json(info_path, info)\n    write_json(Path(weights).parent / "metrics.json", metrics)\n    return metrics\n', 'config/translations.json': '{\n  "AIR COMPRESSOR": "Compresor de aire",\n  "ALTERNATOR": "Alternador",\n  "BATTERY": "Batería",\n  "BRAKE CALIPER": "Pinza de freno",\n  "BRAKE PAD": "Pastilla de freno",\n  "BRAKE ROTOR": "Disco de freno",\n  "CAMSHAFT": "Árbol de levas",\n  "CARBERATOR": "Carburador",\n  "CLUTCH PLATE": "Disco de embrague",\n  "COIL SPRING": "Resorte helicoidal",\n  "CRANKSHAFT": "Cigüeñal",\n  "CYLINDER HEAD": "Culata",\n  "DISTRIBUTOR": "Distribuidor",\n  "ENGINE BLOCK": "Bloque del motor",\n  "ENGINE VALVE": "Válvula del motor",\n  "FUEL INJECTOR": "Inyector de combustible",\n  "FUSE BOX": "Caja de fusibles",\n  "GAS CAP": "Tapa del depósito de combustible",\n  "HEADLIGHTS": "Faros delanteros",\n  "IDLER ARM": "Brazo auxiliar de dirección",\n  "IGNITION COIL": "Bobina de encendido",\n  "INSTRUMENT CLUSTER": "Cuadro de instrumentos",\n  "LEAF SPRING": "Ballesta",\n  "LOWER CONTROL ARM": "Brazo inferior de suspensión",\n  "MUFFLER": "Silenciador de escape",\n  "OIL FILTER": "Filtro de aceite",\n  "OIL PAN": "Cárter de aceite",\n  "OIL PRESSURE SENSOR": "Sensor de presión de aceite",\n  "OVERFLOW TANK": "Depósito de expansión",\n  "OXYGEN SENSOR": "Sensor de oxígeno",\n  "PISTON": "Pistón",\n  "PRESSURE PLATE": "Plato de presión del embrague",\n  "RADIATOR": "Radiador",\n  "RADIATOR FAN": "Ventilador del radiador",\n  "RADIATOR HOSE": "Manguera del radiador",\n  "RADIO": "Radio",\n  "RIM": "Aro de rueda",\n  "SHIFT KNOB": "Pomo de la palanca de cambios",\n  "SIDE MIRROR": "Espejo lateral",\n  "SPARK PLUG": "Bujía",\n  "SPOILER": "Alerón",\n  "STARTER": "Motor de arranque",\n  "TAILLIGHTS": "Luces traseras",\n  "THERMOSTAT": "Termostato",\n  "TORQUE CONVERTER": "Convertidor de par",\n  "TRANSMISSION": "Transmisión",\n  "VACUUM BRAKE BOOSTER": "Servofreno de vacío",\n  "VALVE LIFTER": "Taqué de válvula",\n  "WATER PUMP": "Bomba de agua",\n  "WINDOW REGULATOR": "Mecanismo elevalunas"\n}\n', 'scripts/summarize_dataset.py': '"""Publish a small verified manifest, excluding local paths and CSV image rows."""\nimport json\nimport sys\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom autoparts_ai.dataset import ROOT, write_json\n\nif __name__ == "__main__":\n    manifest = json.loads((ROOT / "artifacts/dataset.json").read_text(encoding="utf-8"))\n    source = manifest["source"]\n    summary = {\n        "schema_version": 1,\n        "source": {key: source.get(key) for key in ("slug", "url", "version", "license", "license_files", "selection_rule", "variants")},\n        "selected_variant": Path(source["selected_path"]).name,\n        "class_count": manifest["class_count"], "total_images": manifest["total_images"],\n        "splits": {split: {"image_count": details["image_count"]} for split, details in manifest["splits"].items()},\n        "classes": manifest["classes"],\n        "csv_metadata": [{key: table[key] for key in ("file", "columns", "row_count")} for table in manifest["csv_metadata"]],\n        "original_files": manifest["original_files"],\n        "cross_split_duplicate_count": len(manifest["cross_split_duplicate_content"]),\n        "reference_images_published": False,\n        "license_note": "No se encontró archivo de licencia en la descarga; pendiente verificar en Kaggle antes de redistribuir imágenes.",\n        "translations_note": "Traducciones manuales revisables, externas a los metadatos originales.",\n    }\n    write_json(ROOT / "config/dataset-summary.json", summary)\n    print(json.dumps({key: value for key, value in summary.items() if key not in {"classes"}}, ensure_ascii=False, indent=2))\n', 'autoparts_ai/__init__.py': '', 'config/train.json': '{\n  "model": "yolo11n-cls.pt",\n  "epochs": 15,\n  "patience": 4,\n  "imgsz": 224,\n  "batch": -1,\n  "device": 0,\n  "workers": 2,\n  "seed": 42,\n  "deterministic": true,\n  "cache": false\n}'}
for relative, contents in EMBEDDED.items():
    target = WORK / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(contents, encoding="utf-8")
sys.path.insert(0, str(WORK))
CONFIG_PATH = WORK / "config/train.json"
TRAIN_CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
# Edita aquí antes de entrenar, por ejemplo: TRAIN_CONFIG["batch"] = 16
CONFIG_PATH.write_text(json.dumps(TRAIN_CONFIG, indent=2), encoding="utf-8")
print(json.dumps(TRAIN_CONFIG, indent=2))


## 4. Descargar e inspeccionar el dataset real

KaggleHub reutiliza su caché. Si Kaggle solicita autenticación, usa `kagglehub.login()` en una celda aparte y su diálogo; nunca pegues claves en este notebook ni en los artefactos.

El código comprueba variantes, clases, imágenes, CSV y particiones. Selecciona la variante coherente con más clases sin mezclar sus conjuntos. El conteo procede de los archivos descargados, no de la URL. No publica imágenes de referencia si la licencia no está verificada.

In [ ]:
from autoparts_ai.dataset import download_dataset

manifest = download_dataset()
print("Versión observable:", manifest["source"]["version"])
print("Variantes:", manifest["source"]["variants"])
print("Clases verificadas:", manifest["class_count"])
print("Imágenes por partición:", {s: d["image_count"] for s, d in manifest["splits"].items()})
print("Duplicados entre particiones:", len(manifest["cross_split_duplicate_content"]))
print("Licencia local disponible:", manifest["source"]["license"])
for entry in manifest["classes"]:
    print(entry["id"], entry["original_name"], "→", entry["spanish_name"], entry["image_counts"])


## 5. Entrenar con train y validación

El entrenador ve una carpeta que contiene físicamente **solo train y val**. Test permanece fuera. Se guarda `best.pt` por fitness de validación junto con la época real del checkpoint y métricas de validación.

La celda puede tardar varios minutos. Su duración depende de la GPU. No cierres Colab durante el proceso. No afirma éxito hasta comprobar que existen pesos.

In [ ]:
from autoparts_ai.training import train

training_info = train(CONFIG_PATH)
WEIGHTS = WORK / "artifacts/models/best.pt"
if not WEIGHTS.is_file() or WEIGHTS.stat().st_size == 0:
    raise RuntimeError("No se generaron pesos entrenados.")
print("Mejor época:", training_info["training"]["best_epoch"])
print("Segundos de entrenamiento:", training_info["training"]["duration_seconds"])
print("Métricas validación:", training_info["validation_metrics"])


## 6. Evaluar una sola vez sobre test

Esta evaluación ocurre después del entrenamiento. Produce Top-1, Top-5, precision/recall/F1 macro, matriz de confusión y métricas por clase. Si hay duplicados entre particiones, registra la advertencia. No uses estos resultados para escoger parámetros; para eso está validación.

In [ ]:
from autoparts_ai.evaluation import evaluate

metrics = evaluate(WEIGHTS, WORK / "artifacts/dataset.json", device=0)
print(json.dumps({k: v for k, v in metrics.items() if k not in {"confusion_matrix", "per_class"}}, indent=2, ensure_ascii=False))
subprocess.run([sys.executable, str(WORK / "scripts/summarize_dataset.py")], check=True)


## 7. Probar una fotografía propia (opcional)

Deja `PROBAR_FOTO = False` para omitir. Si lo activas, sube una fotografía JPG, PNG o WebP. La imagen se procesa en memoria y no se añade al ZIP. Se muestran las tres clases más probables; sus porcentajes no garantizan identificación.

In [ ]:
PROBAR_FOTO = False
if PROBAR_FOTO:
    import io
    from PIL import Image
    from ultralytics import YOLO
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Selecciona una sola fotografía.")
    filename, content = next(iter(uploaded.items()))
    if len(content) > 10 * 1024 * 1024:
        raise ValueError("Imagen demasiado grande: máximo 10 MiB.")
    with Image.open(io.BytesIO(content)) as raw:
        if raw.format not in {"JPEG", "PNG", "WEBP"} or raw.width * raw.height > 20_000_000:
            raise ValueError("Formato o dimensiones de imagen no permitidos.")
        photo = raw.convert("RGB")
    result = YOLO(str(WEIGHTS), task="classify").predict(photo, imgsz=TRAIN_CONFIG["imgsz"], device=0, verbose=False)[0]
    translations = json.loads((WORK / "config/translations.json").read_text(encoding="utf-8"))
    for class_id in result.probs.top5[:3]:
        original = result.names[class_id]
        print(translations.get(original, original), "|", original, "|", round(float(result.probs.data[class_id]) * 100, 2), "%")
    display(photo)
    # files.upload puede crear copia temporal en el directorio Colab: excluida del ZIP.
    del uploaded, content


## 8. Verificar y descargar artefactos reales

El ZIP incluye únicamente pesos, información del modelo, métricas, resumen de clases, traducciones, configuración y versiones de dependencias. No incluye dataset, imágenes, caché ni claves. Se comprueba SHA-256 y la existencia de métricas test antes de permitir la exportación.

Guarda el ZIP antes de que termine la sesión: el disco Colab es temporal. Compártelo para integrar los artefactos en `ai-service/artifacts/models/`. El umbral de confianza del servicio se ajusta posteriormente usando validación; no se infiere de un porcentaje aislado.

In [ ]:
import hashlib
import math
import zipfile
from google.colab import files

MODELS = WORK / "artifacts/models"
info = json.loads((MODELS / "model-info.json").read_text(encoding="utf-8"))
saved_metrics = json.loads((MODELS / "metrics.json").read_text(encoding="utf-8"))
digest = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
if digest != info.get("weights_sha256"):
    raise RuntimeError("SHA-256 de pesos no coincide con model-info.json.")
if saved_metrics.get("split") != "test" or saved_metrics.get("sample_count", 0) <= 0 or info.get("metrics") != saved_metrics:
    raise RuntimeError("Faltan métricas test reales y consistentes.")
for key in ("top1_accuracy", "top5_accuracy", "macro_precision", "macro_recall", "macro_f1"):
    value = saved_metrics.get(key)
    if not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 <= value <= 1:
        raise RuntimeError("Métrica inválida: " + key)
environment = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout
EXPORT = WORK / "autoparts-colab-artifacts.zip"
entries = {
    "best.pt": WEIGHTS,
    "model-info.json": MODELS / "model-info.json",
    "metrics.json": MODELS / "metrics.json",
    "dataset-summary.json": WORK / "config/dataset-summary.json",
    "translations.json": WORK / "config/translations.json",
    "train-config.json": CONFIG_PATH,
}
for name, path in entries.items():
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError("Artefacto ausente: " + name)
with zipfile.ZipFile(EXPORT, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name, path in entries.items():
        archive.write(path, arcname=name)
    archive.writestr("environment.txt", environment)
print("ZIP:", EXPORT.name, "| MiB:", round(EXPORT.stat().st_size / 1024**2, 2))
print("SHA-256 best.pt:", digest)
print("Clases:", info["class_count"], "| Top-1 test:", saved_metrics["top1_accuracy"])
files.download(str(EXPORT))


## 9. Copia opcional a Google Drive

Esta celda no monta Drive automáticamente. Cambia `GUARDAR_EN_DRIVE` solo si deseas guardar una copia y autorizas el acceso mediante el diálogo de Google. El ZIP ya puede descargarse sin Drive.

In [ ]:
GUARDAR_EN_DRIVE = False
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    target = Path("/content/drive/MyDrive/autoparts-ai")
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(EXPORT, target / EXPORT.name)
    print("Copia guardada:", target / EXPORT.name)
